In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SOURCE_LAKEHOUSE   = "Bronze_Production_Lakehouse"
TARGET_LAKEHOUSE   = "Bronze_Production_Lakehouse"
TARGET_SCHEMA      = "dbo"

SNAPSHOT_TABLE     = "jdso_parity_snapshot"   # append — daily field-level summary
ISSUES_TABLE       = "jdso_parity_issues"      # overwrite — current mismatch rows
RUN_LOG_TABLE      = "jdso_parity_run_log"     # append — heartbeat / run metadata

# Alert: set to 0 to alert on any mismatch, or raise threshold to suppress noise
ALERT_THRESHOLD    = 0
# ALERT_WEBHOOK    = "https://..."  # uncomment and set to wire up notifications

In [ ]:
import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

run_ts   = datetime.datetime.utcnow()
run_date = run_ts.date()
print(f"Run date: {run_date}  UTC: {run_ts}")

In [ ]:
# ── Load source tables ────────────────────────────────────────────────────────
equip_contact  = spark.read.table(f"{SOURCE_LAKEHOUSE}.Equip.contact")
armaster       = spark.read.table(f"{SOURCE_LAKEHOUSE}.Equip.ArMaster")
apmaster       = spark.read.table(f"{SOURCE_LAKEHOUSE}.Equip.APMASTER")
wkmechfl       = spark.read.table(f"{SOURCE_LAKEHOUSE}.Equip.WKMECHFL")
vh_salman      = spark.read.table(f"{SOURCE_LAKEHOUSE}.Equip.VhSalman")
dv_contact     = spark.read.table(f"{SOURCE_LAKEHOUSE}.Dataverse.contact")
dv_account     = spark.read.table(f"{SOURCE_LAKEHOUSE}.Dataverse.account")

# Register as temp views for Spark SQL
equip_contact.createOrReplaceTempView("eq_contact")
armaster.createOrReplaceTempView("eq_armaster")
apmaster.createOrReplaceTempView("eq_apmaster")
wkmechfl.createOrReplaceTempView("eq_wkmechfl")
vh_salman.createOrReplaceTempView("eq_vhsalman")
dv_contact.createOrReplaceTempView("dv_contact")
dv_account.createOrReplaceTempView("dv_account")

print("Source tables loaded.")

In [ ]:
# ── Section 0 — Sync coverage ─────────────────────────────────────────────────
# Counts how many active EQUIP contacts have / don't have a matching Dataverse row.

coverage = spark.sql("""
WITH ar_ap AS (
    SELECT UPPER(contact_code) AS cc FROM eq_armaster WHERE contact_code IS NOT NULL
    UNION
    SELECT UPPER(CONTACT_CODE)       FROM eq_apmaster  WHERE CONTACT_CODE IS NOT NULL
),
equip_active AS (
    SELECT UPPER(c.contact_code) AS cc
    FROM eq_contact c
    INNER JOIN ar_ap a ON a.cc = UPPER(c.contact_code)
    LEFT  JOIN eq_wkmechfl m ON m.Code = c.contact_code
    LEFT  JOIN eq_vhsalman s ON s.CODE = c.contact_code
    WHERE IFNULL(c.Inactive_Indicator, 'A') <> 'I'
      AND m.Code IS NULL AND s.CODE IS NULL
),
jdso_active AS (
    SELECT REGEXP_REPLACE(REGEXP_REPLACE(UPPER(dc.jd_referenceid),
               'ARCONTACT-', ''), 'APCONTACT-', '') AS cc
    FROM dv_contact dc
    INNER JOIN ar_ap a
        ON a.cc = REGEXP_REPLACE(REGEXP_REPLACE(UPPER(dc.jd_referenceid),
                      'ARCONTACT-', ''), 'APCONTACT-', '')
    LEFT  JOIN eq_wkmechfl m
        ON UPPER(m.Code) = REGEXP_REPLACE(REGEXP_REPLACE(UPPER(dc.jd_referenceid),
                               'ARCONTACT-', ''), 'APCONTACT-', '')
    LEFT  JOIN eq_vhsalman s
        ON UPPER(s.CODE) = REGEXP_REPLACE(REGEXP_REPLACE(UPPER(dc.jd_referenceid),
                               'ARCONTACT-', ''), 'APCONTACT-', '')
    LEFT  JOIN dv_account ar_acct ON dc.parentcustomerid = ar_acct.Id
    LEFT  JOIN dv_account ap_acct ON dc.jd_vendorid      = ap_acct.Id
    WHERE (dc.jd_referenceid LIKE 'ARCONTACT-%' OR dc.jd_referenceid LIKE 'APCONTACT-%')
      AND dc.statecode = 0
      AND m.Code IS NULL AND s.CODE IS NULL
      AND IFNULL(COALESCE(ar_acct.statuscode, ap_acct.statuscode), 0) <> 2
)
SELECT
    COUNT(e.cc)                                                         AS total_active_equip,
    SUM(CASE WHEN e.cc IS NOT NULL AND j.cc IS NOT NULL THEN 1 ELSE 0 END) AS matched_in_jdso,
    SUM(CASE WHEN e.cc IS NOT NULL AND j.cc IS NULL     THEN 1 ELSE 0 END) AS equip_only,
    SUM(CASE WHEN e.cc IS NULL     AND j.cc IS NOT NULL THEN 1 ELSE 0 END) AS jdso_only
FROM equip_active e
FULL OUTER JOIN jdso_active j ON e.cc = j.cc
""")

coverage.show()
cov = coverage.collect()[0]
total_contacts_checked = cov['total_active_equip']
print(f"Contacts in scope: {total_contacts_checked:,}  |  Matched in JDSO: {cov['matched_in_jdso']:,}  |  EQUIP-only: {cov['equip_only']:,}  |  JDSO-only: {cov['jdso_only']:,}")

In [ ]:
# ── Build joined field comparison view ───────────────────────────────────────
# Used by both the summary (Section 1) and detail (Section 2) cells below.

spark.sql("""
CREATE OR REPLACE TEMP VIEW jdso_joined AS
WITH ar_ap AS (
    SELECT UPPER(contact_code) AS cc FROM eq_armaster WHERE contact_code IS NOT NULL
    UNION
    SELECT UPPER(CONTACT_CODE)       FROM eq_apmaster  WHERE CONTACT_CODE IS NOT NULL
),
active_contacts AS (
    SELECT
        c.contact_code,
        c.Business_Individual,
        NULLIF(TRIM(c.name),           '') AS eq_firstname,
        NULLIF(TRIM(c.initial),        '') AS eq_middlename,
        NULLIF(TRIM(c.surname),        '') AS eq_lastname,
        NULLIF(TRIM(c.Familiar_Name),  '') AS eq_nickname,
        NULLIF(TRIM(c.title),          '') AS eq_salutation,
        NULLIF(TRIM(c.Suffix),         '') AS eq_suffix,
        NULLIF(TRIM(c.Generation),     '') AS eq_generation,
        NULLIF(TRIM(c.company_name),   '') AS eq_company_name,
        NULLIF(TRIM(c.email_address),  '') AS eq_email1,
        NULLIF(TRIM(REGEXP_REPLACE(c.BusinessPhone, '[^0-9]', '')), '') AS eq_phone1,
        NULLIF(TRIM(REGEXP_REPLACE(c.PrivatePhone,  '[^0-9]', '')), '') AS eq_phone2,
        NULLIF(TRIM(REGEXP_REPLACE(c.MobilePhone,   '[^0-9]', '')), '') AS eq_mobile,
        NULLIF(TRIM(REGEXP_REPLACE(c.fax_no,        '[^0-9]', '')), '') AS eq_fax,
        NULLIF(TRIM(CAST(c.Cmp_Ckc_Id AS STRING)),  '') AS eq_cmp_ckc_id,
        NULLIF(TRIM(c.street),          '') AS eq_addr1_line1,
        NULLIF(TRIM(c.street_2),        '') AS eq_addr1_line2,
        NULLIF(TRIM(c.city),            '') AS eq_addr1_city,
        NULLIF(TRIM(c.state),           '') AS eq_addr1_state,
        NULLIF(TRIM(c.pcode),           '') AS eq_addr1_zip,
        NULLIF(TRIM(c.County),          '') AS eq_addr1_county,
        NULLIF(TRIM(c.country),         '') AS eq_addr1_country,
        NULLIF(TRIM(c.postal_street_1), '') AS eq_addr2_line1,
        NULLIF(TRIM(c.postal_street_2), '') AS eq_addr2_line2,
        NULLIF(TRIM(c.postal_city),     '') AS eq_addr2_city,
        NULLIF(TRIM(c.postal_state),    '') AS eq_addr2_state,
        NULLIF(TRIM(c.postal_pcode),    '') AS eq_addr2_zip,
        NULLIF(TRIM(c.Postal_County),   '') AS eq_addr2_county,
        NULLIF(TRIM(c.postal_country),  '') AS eq_addr2_country
    FROM eq_contact c
    INNER JOIN ar_ap a ON a.cc = UPPER(c.contact_code)
    LEFT  JOIN eq_wkmechfl m ON m.Code = c.contact_code
    LEFT  JOIN eq_vhsalman s ON s.CODE = c.contact_code
    WHERE IFNULL(c.Inactive_Indicator, 'A') <> 'I'
      AND m.Code IS NULL AND s.CODE IS NULL
),
jdso_contacts AS (
    SELECT
        REGEXP_REPLACE(REGEXP_REPLACE(UPPER(dc.jd_referenceid),
            'ARCONTACT-', ''), 'APCONTACT-', '') AS contact_code,
        NULLIF(TRIM(dc.firstname),     '') AS dv_firstname,
        NULLIF(TRIM(dc.middlename),    '') AS dv_middlename,
        NULLIF(TRIM(dc.lastname),      '') AS dv_lastname,
        NULLIF(TRIM(dc.nickname),      '') AS dv_nickname,
        NULLIF(TRIM(dc.salutation),    '') AS dv_salutation,
        NULLIF(TRIM(dc.suffix),        '') AS dv_suffix,
        NULLIF(TRIM(dc.jd_generation), '') AS dv_generation,
        NULLIF(TRIM(COALESCE(ar_acct.jd_companyname, ap_acct.jd_companyname)), '') AS dv_company_name,
        NULLIF(TRIM(dc.emailaddress1), '') AS dv_email1,
        NULLIF(TRIM(REGEXP_REPLACE(dc.telephone1, '[^0-9]', '')), '') AS dv_phone1,
        NULLIF(TRIM(REGEXP_REPLACE(dc.telephone2, '[^0-9]', '')), '') AS dv_phone2,
        NULLIF(TRIM(REGEXP_REPLACE(dc.mobilephone,'[^0-9]', '')), '') AS dv_mobile,
        NULLIF(TRIM(REGEXP_REPLACE(dc.fax,        '[^0-9]', '')), '') AS dv_fax,
        NULLIF(TRIM(CAST(dc.jd_compckcid AS STRING)), '') AS dv_cmp_ckc_id,
        NULLIF(TRIM(dc.address1_line1),           '') AS dv_addr1_line1,
        NULLIF(TRIM(dc.address1_line2),           '') AS dv_addr1_line2,
        NULLIF(TRIM(dc.address1_city),            '') AS dv_addr1_city,
        NULLIF(TRIM(dc.address1_stateorprovince), '') AS dv_addr1_state,
        NULLIF(TRIM(dc.address1_postalcode),      '') AS dv_addr1_zip,
        NULLIF(TRIM(dc.address1_county),          '') AS dv_addr1_county,
        NULLIF(TRIM(dc.address1_country),         '') AS dv_addr1_country,
        NULLIF(TRIM(dc.address2_line1),           '') AS dv_addr2_line1,
        NULLIF(TRIM(dc.address2_line2),           '') AS dv_addr2_line2,
        NULLIF(TRIM(dc.address2_city),            '') AS dv_addr2_city,
        NULLIF(TRIM(dc.address2_stateorprovince), '') AS dv_addr2_state,
        NULLIF(TRIM(dc.address2_postalcode),      '') AS dv_addr2_zip,
        NULLIF(TRIM(dc.address2_county),          '') AS dv_addr2_county,
        NULLIF(TRIM(dc.address2_country),         '') AS dv_addr2_country
    FROM dv_contact dc
    INNER JOIN ar_ap a
        ON a.cc = REGEXP_REPLACE(REGEXP_REPLACE(UPPER(dc.jd_referenceid),
                      'ARCONTACT-', ''), 'APCONTACT-', '')
    LEFT  JOIN eq_wkmechfl m
        ON UPPER(m.Code) = REGEXP_REPLACE(REGEXP_REPLACE(UPPER(dc.jd_referenceid),
                               'ARCONTACT-', ''), 'APCONTACT-', '')
    LEFT  JOIN eq_vhsalman s
        ON UPPER(s.CODE) = REGEXP_REPLACE(REGEXP_REPLACE(UPPER(dc.jd_referenceid),
                               'ARCONTACT-', ''), 'APCONTACT-', '')
    LEFT  JOIN dv_account ar_acct ON dc.parentcustomerid = ar_acct.Id
    LEFT  JOIN dv_account ap_acct ON dc.jd_vendorid      = ap_acct.Id
    WHERE (dc.jd_referenceid LIKE 'ARCONTACT-%' OR dc.jd_referenceid LIKE 'APCONTACT-%')
      AND dc.statecode = 0
      AND m.Code IS NULL AND s.CODE IS NULL
      AND IFNULL(COALESCE(ar_acct.statuscode, ap_acct.statuscode), 0) <> 2
)
SELECT
    e.contact_code,
    e.Business_Individual,
    CASE WHEN d.contact_code IS NULL THEN 1 ELSE 0 END AS equip_only,
    e.eq_firstname,    d.dv_firstname,
    e.eq_middlename,   d.dv_middlename,
    e.eq_lastname,     d.dv_lastname,
    e.eq_nickname,     d.dv_nickname,
    e.eq_salutation,   d.dv_salutation,
    e.eq_suffix,       d.dv_suffix,
    e.eq_generation,   d.dv_generation,
    e.eq_company_name, d.dv_company_name,
    e.eq_email1,       d.dv_email1,
    e.eq_phone1,       d.dv_phone1,
    e.eq_phone2,       d.dv_phone2,
    e.eq_mobile,       d.dv_mobile,
    e.eq_fax,          d.dv_fax,
    e.eq_cmp_ckc_id,   d.dv_cmp_ckc_id,
    e.eq_addr1_line1,  d.dv_addr1_line1,
    e.eq_addr1_line2,  d.dv_addr1_line2,
    e.eq_addr1_city,   d.dv_addr1_city,
    e.eq_addr1_state,  d.dv_addr1_state,
    e.eq_addr1_zip,    d.dv_addr1_zip,
    e.eq_addr1_county, d.dv_addr1_county,
    e.eq_addr1_country,d.dv_addr1_country,
    e.eq_addr2_line1,  d.dv_addr2_line1,
    e.eq_addr2_line2,  d.dv_addr2_line2,
    e.eq_addr2_city,   d.dv_addr2_city,
    e.eq_addr2_state,  d.dv_addr2_state,
    e.eq_addr2_zip,    d.dv_addr2_zip,
    e.eq_addr2_county, d.dv_addr2_county,
    e.eq_addr2_country,d.dv_addr2_country
FROM active_contacts e
LEFT JOIN jdso_contacts d ON UPPER(e.contact_code) = d.contact_code
""")

print("jdso_joined view created.")

In [ ]:
# ── Section 1 — Field-level summary (append to snapshot table) ────────────────

FIELDS = [
    "firstname", "middlename", "lastname", "nickname", "salutation",
    "suffix", "generation", "company_name", "email1",
    "phone1", "phone2", "mobile", "fax", "cmp_ckc_id",
    "addr1_line1", "addr1_line2", "addr1_city", "addr1_state", "addr1_zip",
    "addr1_county", "addr1_country",
    "addr2_line1", "addr2_line2", "addr2_city", "addr2_state", "addr2_zip",
    "addr2_county", "addr2_country",
]

union_parts = " UNION ALL ".join([
    f"""SELECT '{f}' AS field_name, equip_only,
               eq_{f} AS eq_val, dv_{f} AS dv_val FROM jdso_joined"""
    for f in FIELDS
])

summary_sdf = spark.sql(f"""
WITH fc AS ({union_parts})
SELECT
    field_name,
    SUM(CASE WHEN equip_only = 0 THEN 1 ELSE 0 END) AS matched_contacts,
    SUM(CASE WHEN equip_only = 0
             AND eq_val IS NOT NULL AND dv_val IS NOT NULL
             AND UPPER(eq_val) <> UPPER(dv_val) THEN 1 ELSE 0 END) AS conflict,
    SUM(CASE WHEN equip_only = 0
             AND eq_val IS NOT NULL AND dv_val IS NULL THEN 1 ELSE 0 END) AS equip_ahead,
    SUM(CASE WHEN equip_only = 0
             AND eq_val IS NULL AND dv_val IS NOT NULL THEN 1 ELSE 0 END) AS jdso_ahead
FROM fc
GROUP BY field_name
ORDER BY conflict DESC
""")

summary_sdf = (
    summary_sdf
    .withColumn("snapshot_date", F.lit(str(run_date)))
    .withColumn("conflict_pct",
        F.round(
            F.col("conflict") * 100.0
            / F.when(F.col("matched_contacts") != 0, F.col("matched_contacts")),
            1
        ))
    .select("snapshot_date", "field_name", "matched_contacts",
            "conflict", "equip_ahead", "jdso_ahead", "conflict_pct")
)

summary_sdf.show(30, truncate=False)

(
    summary_sdf.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{TARGET_LAKEHOUSE}.{TARGET_SCHEMA}.{SNAPSHOT_TABLE}")
)
print(f"Written to {TARGET_LAKEHOUSE}.{TARGET_SCHEMA}.{SNAPSHOT_TABLE} (append)")

In [ ]:
# ── Section 2 — Per-contact mismatch detail (overwrite issues table) ──────────

detail_parts = " UNION ALL ".join([
    f"""SELECT contact_code, Business_Individual, '{f}' AS field_name,
               eq_{f} AS equip_value, dv_{f} AS jdso_value
        FROM jdso_joined
        WHERE equip_only = 0
          AND (
               (eq_{f} IS NOT NULL AND dv_{f} IS NOT NULL AND UPPER(eq_{f}) <> UPPER(dv_{f}))
            OR (eq_{f} IS NOT NULL AND dv_{f} IS NULL)
            OR (eq_{f} IS NULL     AND dv_{f} IS NOT NULL)
          )"""
    for f in FIELDS
])

issues_sdf = spark.sql(f"""
SELECT
    contact_code,
    Business_Individual,
    field_name,
    equip_value,
    jdso_value,
    CASE
        WHEN equip_value IS NOT NULL AND jdso_value IS NOT NULL
             AND UPPER(equip_value) <> UPPER(jdso_value) THEN 'CONFLICT'
        WHEN equip_value IS NOT NULL AND jdso_value IS NULL  THEN 'EQUIP_AHEAD'
        WHEN equip_value IS NULL     AND jdso_value IS NOT NULL THEN 'JDSO_AHEAD'
    END AS mismatch_type
FROM ({detail_parts}) t
ORDER BY contact_code, field_name
""")

total_mismatch_rows = issues_sdf.count()
print(f"Total mismatch rows: {total_mismatch_rows:,}")

(
    issues_sdf.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{TARGET_LAKEHOUSE}.{TARGET_SCHEMA}.{ISSUES_TABLE}")
)
print(f"Written to {TARGET_LAKEHOUSE}.{TARGET_SCHEMA}.{ISSUES_TABLE} (overwrite)")

In [ ]:
# ── Run log (heartbeat) ───────────────────────────────────────────────────────
# Appended every run so you can confirm the pipeline ran even when mismatch rows = 0.

from pyspark.sql import Row

log_row = Row(
    run_date=str(run_date),
    run_timestamp=str(run_ts),
    total_contacts_checked=total_contacts_checked,
    total_mismatch_rows=total_mismatch_rows,
    status="SUCCESS",
)
log_sdf = spark.createDataFrame([log_row])

(
    log_sdf.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{TARGET_LAKEHOUSE}.{TARGET_SCHEMA}.{RUN_LOG_TABLE}")
)
print(f"Run log written. Contacts checked: {total_contacts_checked:,} | Mismatch rows: {total_mismatch_rows:,}")

In [ ]:
# ── Alert stub ────────────────────────────────────────────────────────────────
# Fires when mismatch rows exceed ALERT_THRESHOLD.
# Wire up ALERT_WEBHOOK (Power Automate HTTP trigger, Teams, etc.) to activate.

if total_mismatch_rows > ALERT_THRESHOLD:
    msg = (
        f"JDSO parity check {run_date}: {total_mismatch_rows:,} mismatch rows detected "
        f"across {total_contacts_checked:,} contacts. "
        f"Query {TARGET_LAKEHOUSE}.{TARGET_SCHEMA}.{ISSUES_TABLE} to investigate."
    )
    print(f"ALERT: {msg}")
    # Uncomment to send via Power Automate / Teams webhook:
    # import requests
    # requests.post(ALERT_WEBHOOK, json={"text": msg}, timeout=10)
else:
    print(f"No alert — mismatch rows ({total_mismatch_rows:,}) at or below threshold ({ALERT_THRESHOLD}).")